# 0. Импорт и конфигурация

In [ ]:
RANDOM_STATE = 42

import os
import math
import json
import numpy as np
import pandas as pd
from typing import List, Iterable, Tuple

def set_global_seed(seed: int = RANDOM_STATE) -> None:
    np.random.seed(seed)

set_global_seed(RANDOM_STATE)

# 1. Обработка данных

In [ ]:
import pandas as pd


df = pd.read_csv('train_540k.csv')


In [ ]:

df_base = df.copy()
df_base = df_base.drop(columns=['id'])

for col in df_base.columns:
    if df_base[col].isnull().any():
        df_base[col] = df_base[col].fillna(df_base[col].median())

cat_cols = [col for col in df_base.columns if '_cat' in col]
for col in cat_cols:
    if (df_base[col] == -1).any():
        mode_val = df_base[df_base[col] != -1][col].mode()[0]
        df_base[col] = df_base[col].replace(-1, mode_val)

In [ ]:
from sklearn.preprocessing import StandardScaler

df_processed = df.copy()
df_processed = df_processed.drop(columns=['id'])

for col in df_processed.columns:
    if df_processed[col].isnull().any():
        df_processed[col] = df_processed[col].fillna(df_processed[col].median())

cat_cols = [col for col in df_processed.columns if '_cat' in col]
for col in cat_cols:
    if (df_processed[col] == -1).any():
        mode_val = df_processed[df_processed[col] != -1][col].mode()[0]
        df_processed[col] = df_processed[col].replace(-1, mode_val)

calc_cols = [col for col in df_processed.columns if 'calc' in col]
df_processed = df_processed.drop(columns=calc_cols)

# Стандартизация
numeric_cols = [col for col in df_processed.columns
                if col != 'target' and '_bin' not in col and '_cat' not in col]

scaler_features = StandardScaler()
df_processed[numeric_cols] = scaler_features.fit_transform(df_processed[numeric_cols])


## Разделение на train/val/test

In [ ]:
X_base, y_base = df_base.drop(columns=['target']), df_base['target']
X_processed, y_processed = df_processed.drop(columns=['target']), df_processed['target']



In [ ]:

from sklearn.model_selection import train_test_split

X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_base, y_base, test_size=0.2, random_state=42, stratify=y_base
)

In [ ]:

X_temp, X_test_processed, y_temp, y_test_processed = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)

X_train_processed, X_val_processed, y_train_processed, y_val_processed = train_test_split(
    X_temp, y_temp, test_size=0.2, random_state=42, stratify=y_temp
)

# 2. Реализация метрик

In [ ]:
def accuracy_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    correct = np.sum(y_true == y_pred)
    total = len(y_true)
    return correct / total

In [ ]:
def precision_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    if tp + fp == 0:
        return 0.0
    return tp / (tp + fp)

In [ ]:
def recall_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    tp = np.sum((y_true == 1) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    if tp + fn == 0:
        return 0.0
    return tp / (tp + fn)

In [ ]:
def f1_manual(y_true: Iterable[int], y_pred: Iterable[int]) -> float:
    p = precision_manual(y_true, y_pred)
    r = recall_manual(y_true, y_pred)
    if p + r == 0:
        return 0.0
    return 2 * p * r / (p + r)

In [ ]:
def precision_recall_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    thresholds = np.unique(y_score)
    thresholds = np.sort(thresholds)[::-1]

    precisions = []
    recalls = []
    threshold_list = []

    for thresh in thresholds:
        y_pred = (y_score >= thresh).astype(int)

        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))
        fn = np.sum((y_true == 1) & (y_pred == 0))

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

        precisions.append(precision)
        recalls.append(recall)
        threshold_list.append(thresh)

    precisions.append(1.0)
    recalls.append(0.0)

    return np.array(precisions), np.array(recalls), np.array(threshold_list)

In [ ]:
def roc_curve_manual(y_true: Iterable[int], y_score: Iterable[float]) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    y_true = np.array(y_true)
    y_score = np.array(y_score)

    thresholds = np.unique(y_score)
    thresholds = np.sort(thresholds)[::-1]

    total_positives = np.sum(y_true == 1)
    total_negatives = np.sum(y_true == 0)

    fpr_list = [0.0]
    tpr_list = [0.0]
    threshold_list = [thresholds[0] + 1]

    for thresh in thresholds:
        y_pred = (y_score >= thresh).astype(int)

        tp = np.sum((y_true == 1) & (y_pred == 1))
        fp = np.sum((y_true == 0) & (y_pred == 1))

        tpr = tp / total_positives if total_positives > 0 else 0.0
        fpr = fp / total_negatives if total_negatives > 0 else 0.0

        fpr_list.append(fpr)
        tpr_list.append(tpr)
        threshold_list.append(thresh)

    return np.array(fpr_list), np.array(tpr_list), np.array(threshold_list)


In [ ]:
def roc_auc_manual(fpr: Iterable[float], tpr: Iterable[float]) -> float:

    fpr = np.array(fpr)
    tpr = np.array(tpr)

    sorted_indices = np.argsort(fpr)
    fpr_sorted = fpr[sorted_indices]
    tpr_sorted = tpr[sorted_indices]

    auc = np.trapz(tpr_sorted, fpr_sorted)
    return float(auc)


## Бейзлайн

In [ ]:

from sklearn.linear_model import LogisticRegression

logreg = LogisticRegression(max_iter=1000, random_state=42)
logreg.fit(X_train_base, y_train_base)

y_pred_logreg = logreg.predict(X_test_base)
print(f"Logistic Regression Accuracy: {accuracy_manual(y_test_base, y_pred_logreg):.4f}")


Logistic Regression Accuracy: 0.9635


In [ ]:


from sklearn.neighbors import KNeighborsClassifier

sample_size = min(10000, len(X_train_processed))
indices = np.random.choice(len(X_train_processed), sample_size, replace=False)

if isinstance(X_train_processed, pd.DataFrame):
    X_train_knn = X_train_processed.iloc[indices]
    y_train_knn = y_train_processed.iloc[indices]
else:
    X_train_knn = X_train_processed[indices]
    y_train_knn = y_train_processed[indices]

knn = KNeighborsClassifier(n_neighbors=5, algorithm='ball_tree', n_jobs=-1)
knn.fit(X_train_knn, y_train_knn)

y_pred_knn = knn.predict(X_test_processed)
print(f"KNN Accuracy: {accuracy_manual(y_test_processed, y_pred_knn):.4f}")

KNN Accuracy: 0.9631


In [ ]:

from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=42, max_depth=10)
tree.fit(X_train_processed, y_train_processed)

y_pred_tree = tree.predict(X_test_processed)
print(f"Decision Tree Accuracy: {accuracy_manual(y_test_processed, y_pred_tree):.4f}")


Decision Tree Accuracy: 0.9627


In [ ]:

from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=20, random_state=42, max_depth=8, n_jobs=-1)
forest.fit(X_train_processed, y_train_processed)

y_pred_forest = forest.predict(X_test_processed)
print(f"Random Forest Accuracy: {accuracy_manual(y_test_processed, y_pred_forest):.4f}")

Random Forest Accuracy: 0.9635


In [ ]:

from sklearn.svm import LinearSVC

svc = LinearSVC(random_state=42, max_iter=1000, dual=False)
svc.fit(X_train_processed, y_train_processed)

y_pred_svm = svc.predict(X_test_processed)
print(f"SVM Accuracy: {accuracy_manual(y_test_processed, y_pred_svm):.4f}")

SVM Accuracy: 0.9635


# Ансамблирование

In [ ]:
from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

from sklearn.ensemble import VotingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

voting = VotingClassifier(
    estimators=[
        ('lr', LogisticRegression(max_iter=500, random_state=42, solver='lbfgs')),
        ('tree', DecisionTreeClassifier(random_state=42, max_depth=8)),
        ('rf', RandomForestClassifier(n_estimators=10, random_state=42, max_depth=6, n_jobs=-1))
    ],
    voting='hard',
    n_jobs=-1
)

voting.fit(X_train_processed, y_train_processed)

y_pred_voting = voting.predict(X_test_processed)
print(f"Voting Ensemble Accuracy: {accuracy_manual(y_test_processed, y_pred_voting):.4f}")

y_pred_processed = y_pred_forest

Voting Ensemble Accuracy: 0.9635


In [ ]:
from sklearn.metrics import accuracy_score, classification_report

print("=" * 50)
print("СРАВНЕНИЕ МОДЕЛЕЙ")
print("=" * 50)

models_results = {
    'Logistic Regression (baseline)': accuracy_score(y_test_base, y_pred_logreg),
    'KNN': accuracy_score(y_test_processed, y_pred_knn),
    'Decision Tree': accuracy_score(y_test_processed, y_pred_tree),
    'Random Forest': accuracy_score(y_test_processed, y_pred_forest),
    'SVM (Linear)': accuracy_score(y_test_processed, y_pred_svm),
    'Voting Ensemble': accuracy_score(y_test_processed, y_pred_voting)
}

for name, acc in models_results.items():
    print(f"{name:30s}: {acc:.4f}")

СРАВНЕНИЕ МОДЕЛЕЙ
Logistic Regression (baseline): 0.9635
KNN                           : 0.9631
Decision Tree                 : 0.9627
Random Forest                 : 0.9635
SVM (Linear)                  : 0.9635
Voting Ensemble               : 0.9635
